In [1]:
import time
from groq import Groq

In [ ]:
import pandas as pd
from groq import Groq
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

# API in this
client = Groq(api_key="-")

# Hugging Face IMDB dataset
    splits = {'train': 'plain_text/train-00000-of-00001.parquet', 'test': 'plain_text/test-00000-of-00001.parquet'}
    data_train = pd.read_parquet("hf://datasets/stanfordnlp/imdb/" + splits["train"])
    data_test = pd.read_parquet("hf://datasets/stanfordnlp/imdb/" + splits["test"])
data = pd.concat([data_train, data_test], ignore_index=True)

# part of 50.000
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
veri_listesi = data['text'].tolist()[:10000]

def uretim_yap(index, text):
    while True:
        try:
            completion = client.chat.completions.create(
                model="llama-3.1-8b-instant", 
                messages=[
                    {"role": "system", "content": "You are a creative IMDB user. Respond ONLY with the review text. No intros, no explanations."},
                    {"role": "user", "content": f"Write one new review with a similar emotional state and context to the one below, but with a COMPLETELY DIFFERENT scenario. \n\nOriginal: {text}"}
                ],
                temperature=0.7,
                max_tokens=1500,
            )
            return index, completion.choices[0].message.content.strip()
            
        except Exception as e:
            hata = str(e)
            if "429" in hata or "rate limit" in hata.lower():
                # Limited have
                print("limite takıldı")
                
                time.sleep(15)
            else:
                return index, f"HATA: {hata}"

# paralel token creating
print(f"Toplam {len(veri_listesi)} ")

sentetik_metinler = [""] * len(veri_listesi)

# max_workers=20 paralel worker value
with ThreadPoolExecutor(max_workers=10) as executor:
    gelecek_gorevler = {executor.submit(uretim_yap, i, text): i for i, text in enumerate(veri_listesi)}
    
    # counter bar parts
    for sayac, future in enumerate(tqdm(as_completed(gelecek_gorevler), total=len(veri_listesi)), 1):
        idx, sonuc = future.result()
        sentetik_metinler[idx] = sonuc
        if sayac % 500 == 0:
            pd.DataFrame({'orijinal': veri_listesi, 'sentetik': sentetik_metinler}).to_csv(f'groq_yedek_{sayac}.csv', index=False)


final_df = pd.DataFrame({'orijinal': veri_listesi, 'sentetik': sentetik_metinler})
final_df.to_csv('groq_sentetik_50000_TAMAMI.csv', index=False)
print("50.000 verilik dev operasyon başarıyla tamamlandı!")

Toplam 10000 veri 20 şeritli otobanda işleniyor...


  7%|█████▍                                                                        | 691/10000 [00:40<19:32,  7.94it/s]

limite takıldı
limite takıldı


  7%|█████▌                                                                        | 713/10000 [00:43<20:01,  7.73it/s]

limite takıldı


...

limite takıldı
limite takıldı


100%|████████████████████████████████████████████████████████████████████████████▋| 9967/10000 [23:44<00:04,  7.30it/s]

limite takıldı


100%|████████████████████████████████████████████████████████████████████████████| 10000/10000 [24:01<00:00,  6.94it/s]


50.000 verilik dev operasyon başarıyla tamamlandı!


In [6]:
print(data.head())
data=data.tolist()

                                                text  label
0  Forget what I said about Emeril. Rachael Ray i...      0
1  Former private eye-turned-security guard ditch...      0
2  Mann photographs the Alberta Rocky Mountains i...      0
3  Simply put: the movie is boring. Cliché upon c...      0
4  Now being a fan of sci fi, the trailer for thi...      1


AttributeError: 'DataFrame' object has no attribute 'tolist'

In [7]:
for a in range(data.len):
    if a==1:
        if data["label"][a]!=data["label"][a-1]:
            print(a)

AttributeError: 'DataFrame' object has no attribute 'len'

In [2]:
!pip install datasets

   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/527.0 kB ? eta -:--:--
   ---------------------------------------- 527.0/527.0 kB 1.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
    --------------------------------------- 0.5/27.6 MB 3.4 MB/s eta 0:00:09
   - -------------------------------------- 1.0/27.6 MB 2.5 MB/s eta 0:00:11
   --- ------------------------------------ 2.1/27.6 MB 3.3 MB/s eta 0:00:08
   --- ------------------------------------ 2.6/27.6 MB 3.3 MB/s eta 0:00:08
   ---- ----------------------------------- 3.1/27.6 MB 3.0 MB/s eta 0:00:09
   ------ --------------------------------- 4.2/27.6 MB 3.3 MB/s eta 0:00:08
   ------- -------------------------------- 5.2/27.6 MB 3.6 MB/s eta 0:00:07
   --------- ------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you have pandas 3.0.1 which is incompatible.


In [ ]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
#from datasets import load_dataset

print("1. Orijinal IMDB Veri Seti indiriliyor...")
splits = {'train': 'plain_text/train-00000-of-00001.parquet', 'test': 'plain_text/test-00000-of-00001.parquet'}
data_train = pd.read_parquet("hf://datasets/stanfordnlp/imdb/" + splits["train"])
data_test = pd.read_parquet("hf://datasets/stanfordnlp/imdb/" + splits["test"])
df_orijinal = pd.concat([data_train, data_test], ignore_index=True)


df_orijinal = df_orijinal.drop_duplicates(subset=['text'])

print("2. Sentetik Veriler")
df_sentetik = pd.read_csv('groq_sentetik_50000_TAMAMI.csv')
df_orijinal['text'] = df_orijinal['text'].str.strip()
df_sentetik['orijinal'] = df_sentetik['orijinal'].str.strip()

df_final = pd.merge(
    left=df_sentetik, 
    right=df_orijinal[['text', 'label']], 
    left_on='orijinal', 
    right_on='text', 
    how='inner'
)

df_final = df_final.drop(columns=['text'])
df_final.to_csv('groq_sentetik_10000_ETIKETLI.csv', index=False)

print(f"Toplam Eşleşen Satır Sayısı: {len(df_final)}")
print(df_final.head(3))

1. Orijinal IMDB Veri Seti indiriliyor...


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - The pyarrow installation is not built with support for the Parquet file format (DLL load failed while importing _parquet: Belirtilen modül bulunamadı.)
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [ ]:
import pandas as pd

print("1. Orijinal IMDB Veri Seti indiriliyor (CSV Formatında)...")
url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
df_orijinal = pd.read_csv(url)

df_orijinal = df_orijinal.rename(columns={'review': 'text', 'sentiment': 'label'})

df_orijinal['label'] = df_orijinal['label'].map({'positive': 1, 'negative': 0})

df_orijinal = df_orijinal.drop_duplicates(subset=['text'])
df_sentetik = pd.read_csv('groq_sentetik_50000_TAMAMI.csv') 

df_orijinal['text'] = df_orijinal['text'].str.strip()
df_sentetik['orijinal'] = df_sentetik['orijinal'].str.strip()

df_final = pd.merge(
    left=df_sentetik, 
    right=df_orijinal[['text', 'label']], 
    left_on='orijinal', 
    right_on='text', 
    how='inner'
)

df_final = df_final.drop(columns=['text'])
df_final.to_csv('groq_sentetik_10000_ETIKETLI.csv', index=False)

print(f"Toplam Eşleşen Satır Sayısı:{len(df_final)}")
print(df_final.head(3))


1. Orijinal IMDB Veri Seti indiriliyor (CSV Formatında)...
2. Sentetik Verileriniz yükleniyor...
3. Etiketler (Labels) eşleştiriliyor...

--- İŞLEM BAŞARILI! ---
Toplam Eşleşen Satır Sayısı: 10000

İlk 3 Satırın Görünümü:
                                            orijinal  \
0  Forget what I said about Emeril. Rachael Ray i...   
1  Former private eye-turned-security guard ditch...   
2  Mann photographs the Alberta Rocky Mountains i...   

                                            sentetik  label  
0  I just watched an entire season of American Id...      0  
1  A reclusive avant-garde composer, struggling t...      0  
2  Taymor photographs the Japanese city of Kyoto ...      0  
